# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates the parsing of synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr, col

os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC_Simulation") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Synthetic X12 EDI records
edi_data = [("ISA*00*          *00*          *ZZ*1234567890     *ZZ*0987654321     *190101*1200*U*00501*000000001*0*T*:~",),
            ("GS*HC*1234567890*0987654321*20190101*1200*1*X*005010X222A1~",)]

edi_df = spark.createDataFrame(edi_data, ["raw_line"])

edi_bronze_df = edi_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("SYNTHETIC_X12")) \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumnRenamed("raw_line", "raw_payload_json")

edi_bronze_df.write.format("iceberg").mode("append").save("local.bronze_db.raw_payloads")
print("X12 batch ingestion simulation complete.")


In [ ]:
# Synthetic Debezium CDC records
cdc_data = [("{\"payload\": {\"op\": \"c\", \"after\": \"{\\\"id\\\": 1, \\\"status\\\": \\\"active\\\"}\"}}",),
            ("{\"payload\": {\"op\": \"u\", \"after\": \"{\\\"id\\\": 1, \\\"status\\\": \\\"inactive\\\"}\"}}",)]

cdc_df = spark.createDataFrame(cdc_data, ["raw_payload_json"])

cdc_bronze_df = cdc_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("SYNTHETIC_CDC")) \
    .withColumn("payload_id", expr("uuid()"))

cdc_bronze_df.select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json") \
    .write.format("iceberg").mode("append").save("local.bronze_db.raw_payloads")
print("CDC batch ingestion simulation complete.")


In [ ]:
# Verify Bronze Table
spark.sql("SELECT * FROM local.bronze_db.raw_payloads WHERE source_system_id IN ('SYNTHETIC_X12', 'SYNTHETIC_CDC')").show(truncate=False)
